# MTSamples HTML Parser

Parses the per-record HTML files from the Kaggle `mtsamples-dataset` (shoubhikchakra/mtsamples-dataset) into clean structured rows, and extracts medication/plan-relevant sentences for NER annotation.

**Input:** a folder of `N.html` files, e.g. `mtsamples/content/mtsamples/0.html`, `1.html`, ...

**Output:**
- `parsed_mtsamples.csv` — one row per report (specialty, sample name, description, all sections)
- `ner_candidate_sentences.csv` — sentence-level rows from medication/plan sections, ready for Label Studio


## 1. Install & import dependencies

In [4]:
!pip install notebook jupyterlab beautifulsoup4 pandas nbformat


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: C:\Users\rohit\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [5]:
!pip install beautifulsoup4


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: C:\Users\rohit\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [8]:
%pip install beautifulsoup4 pandas nbformat

  Using cached pandas-3.0.3-cp315-cp315-win_amd64.whl
     ---------------------------------------- 0.0/20.7 MB ? eta -:--:--
     - -------------------------------------- 0.5/20.7 MB 3.3 MB/s eta 0:00:07
     - -------------------------------------- 0.8/20.7 MB 1.9 MB/s eta 0:00:11
     -- ------------------------------------- 1.3/20.7 MB 2.0 MB/s eta 0:00:10
     --- ------------------------------------ 1.8/20.7 MB 2.3 MB/s eta 0:00:09
     ---- ----------------------------------- 2.1/20.7 MB 1.9 MB/s eta 0:00:10
     ----- ---------------------------------- 2.6/20.7 MB 2.1 MB/s eta 0:00:09
     ------ --------------------------------- 3.4/20.7 MB 2.3 MB/s eta 0:00:08
     --------- ------------------------------ 4.7/20.7 MB 2.8 MB/s eta 0:00:06
     ----------- ---------------------------- 6.0/20.7 MB 3.2 MB/s eta 0:00:05
     ------------- -------------------------- 6.8/20.7 MB 3.3 MB/s eta 0:00:05
     --------------- ------------------------ 8.1/20.7 MB 3.5 MB/s eta 0:00:04
     


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!python.exe -m pip install --upgrade pip

  Using cached pip-26.1.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.1.1
    Uninstalling pip-26.1.1:
      Successfully uninstalled pip-26.1.1


In [10]:
import csv
import re
from pathlib import Path
from bs4 import BeautifulSoup

## 2. Set your paths

Edit `INPUT_DIR` below to point at your local folder of HTML files.

In [11]:
INPUT_DIR = r"C:\Users\rohit\Downloads\mtsamples\content\mtsamples"
OUTPUT_CSV = "parsed_mtsamples.csv"
SENTENCES_CSV = "ner_candidate_sentences.csv"
LIMIT = 200  # set to None to process all files once you've sanity-checked the output

## 3. Define the high-value sections and helpers

In [12]:
# Sections that are high-value for medication/dosage/instruction NER.
# We keep these; everything else (SUBJECTIVE/OBJECTIVE/HISTORY/etc.) is
# still saved in the full record but excluded from the NER candidate pool
# since it's mostly narrative prose with low drug-entity density.
HIGH_VALUE_SECTIONS = {
    "MEDICATIONS",
    "DISCHARGE MEDICATIONS",
    "CURRENT MEDICATIONS",
    "PLAN",
    "TREATMENT",
    "RECOMMENDATIONS",
    "PRESCRIPTIONS",
    "ASSESSMENT AND PLAN",
    "DISPOSITION",
}

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

In [ ]:
def parse_record(html_path: Path) -> dict:
    raw = html_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(raw, "html.parser")

    record = {
        "file": html_path.name,
        "specialty": "",
        "sample_name": "",
        "description": "",
        "sections": {},
    }

    # --- Structural extraction for specialty / sample name / description ---
    h1 = soup.find("h1")
    if h1:
        h1_text = h1.get_text(" ", strip=True).replace("\xa0", " ")
        m = re.search(r"Medical Specialty:\s*(.+?)\s*Sample Name:", h1_text)
        if m:
            record["specialty"] = m.group(1).strip()
        m = re.search(r"Sample Name:\s*(.+)$", h1_text)
        if m:
            record["sample_name"] = m.group(1).strip()

    h2 = soup.find("h2")
    if h2:
        h2_text = h2.get_text(" ", strip=True).replace("\xa0", " ")
        m = re.search(r"Description:\s*(.+?)(\(Medical Transcription|$)", h2_text)
        if m:
            record["description"] = m.group(1).strip()

    # --- Section body extraction (unchanged: flattened text, ALL-CAPS headers) ---
    text = soup.get_text("\n").replace("\xa0", " ").replace("Â", "")
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    full_text = "\n".join(lines)

    header_re = re.compile(r"^([A-Z][A-Z /]+):\s*(.*)$")
    known = set(re.findall(r"^([A-Z][A-Z /]{2,40}):", full_text, flags=re.MULTILINE))
    current_section, buffer = None, []
    for line in lines:
        hm = header_re.match(line)
        if hm and hm.group(1) in known:
            if current_section:
                record["sections"][current_section] = " ".join(buffer).strip()
            current_section = hm.group(1)
            buffer = [hm.group(2)] if hm.group(2) else []
        else:
            if current_section:
                buffer.append(line)
    if current_section:
        record["sections"][current_section] = " ".join(buffer).strip()

    return record


def _known_headers(full_text: str):
    """Heuristic: treat any short ALL-CAPS line ending in ':' as a header."""
    return set(re.findall(r"^([A-Z][A-Z /]{2,40}):", full_text, flags=re.MULTILINE))


def extract_ner_candidate_sentences(record: dict) -> list:
    """Pull sentences from high-value sections for annotation."""
    candidates = []
    for section, text in record["sections"].items():
        if section in HIGH_VALUE_SECTIONS and text:
            for sent in SENTENCE_SPLIT_RE.split(text):
                sent = sent.strip()
                if len(sent.split()) >= 4:  # skip fragments
                    candidates.append({
                        "file": record["file"],
                        "specialty": record["specialty"],
                        "section": section,
                        "sentence": sent,
                    })
    return candidates

## 4. Run the parser over all HTML files

In [14]:
input_dir = Path(INPUT_DIR)
html_files = sorted(input_dir.glob("*.html"), key=lambda p: int(p.stem) if p.stem.isdigit() else 0)
if LIMIT:
    html_files = html_files[:LIMIT]

print(f"Found {len(html_files)} HTML files to process.")

Found 200 HTML files to process.


In [15]:
all_records = []
all_sentences = []

for hp in html_files:
    try:
        rec = parse_record(hp)
    except Exception as e:
        print(f"  ! skipped {hp.name}: {e}")
        continue
    all_records.append(rec)
    all_sentences.extend(extract_ner_candidate_sentences(rec))

print(f"Parsed {len(all_records)} records.")
print(f"Extracted {len(all_sentences)} NER-candidate sentences from high-value sections.")

Parsed 200 records.
Extracted 373 NER-candidate sentences from high-value sections.


## 5. Write output CSVs

In [16]:
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["file", "specialty", "sample_name", "description", "sections"])
    for rec in all_records:
        writer.writerow([
            rec["file"], rec["specialty"], rec["sample_name"], rec["description"],
            " | ".join(f"{k}::{v}" for k, v in rec["sections"].items()),
        ])

with open(SENTENCES_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["file", "specialty", "section", "sentence"])
    for row in all_sentences:
        writer.writerow([row["file"], row["specialty"], row["section"], row["sentence"]])

print(f"-> {OUTPUT_CSV}")
print(f"-> {SENTENCES_CSV}")

-> parsed_mtsamples.csv
-> ner_candidate_sentences.csv


## 6. Quick sanity check

In [17]:
import pandas as pd

df_sentences = pd.read_csv(SENTENCES_CSV)
print(df_sentences.shape)
df_sentences.head(15)

(373, 4)


,file,specialty,section,sentence
0,0.html,NaN,MEDICATIONS,Her only medication currently is Ortho Tri-Cyc...
1,0.html,NaN,PLAN,She will try Zyrtec instead of Allegra again.
2,0.html,NaN,PLAN,Another option will be to use loratadine.
3,0.html,NaN,PLAN,She does not think she has prescription covera...
4,0.html,NaN,PLAN,Samples of Nasonex two sprays in each nostril ...
5,0.html,NaN,PLAN,A prescription was written as well.
6,1.html,NaN,CURRENT MEDICATIONS,"Atenolol, sodium bicarbonate, Lovaza, and Dial..."
7,1.html,NaN,RECOMMENDATIONS,RAST allergy testing for both food and environ...
8,1.html,NaN,RECOMMENDATIONS,If there is any specific food or inhalant alle...
9,1.html,NaN,RECOMMENDATIONS,"At this point, I would recommend the patient s..."


In [18]:
df_sentences["section"].value_counts()

section
MEDICATIONS              118
RECOMMENDATIONS           80
PLAN                      72
ASSESSMENT AND PLAN       56
DISCHARGE MEDICATIONS     22
TREATMENT                 19
CURRENT MEDICATIONS        3
DISPOSITION                3
Name: count, dtype: int64